# Day 035 · 时间索引与重采样 · 中国版
**DatetimeIndex** · 阶段 P2 · Python 量化工具栈

> 今天讲 Pandas 处理金融数据的看家本领 — 时间索引与重采样。为什么单拎出来讲?因为时间是金融数据的灵魂。股票数据跟普通表格最大的不同,就是它的每一行都死死绑定着一个时间点 — 几月几号、几点几分。把时间这个维度玩明白了,你才算真正会处理行情数据。这一节做五件事 — 第一,讲 DatetimeIndex,Pandas 专门为时间设计的索引,它能让你写一句「给我 12 月的数据」就自动筛出整个 12 月;第二,讲 pd.to_datetime,怎么把行情软件导出的五花八门的日期字符串,统一解析成标准时间戳;第三,讲今天的主角 resample 重采样 — 它能把5 分钟的高频数据,聚合成日线、周线、月线,这是量化里天天用的操作;第四,讲 resample 里最容易踩的坑 — 不同的列要用不同的聚合方式,开盘价取第一个、最高价取最大、成交量要求和,绝不能一刀切全取平均;第五,讲时区处理,A 股是北京时间、美股是纽约时间,做跨市场分析时怎么把它们对齐到同一时刻。我们用真实的茅台5 分钟线,把分钟数据亲手聚合成日线,所有操作都在真数据上跑一遍。

---

### 关于「中国版」

本 notebook 是为**国内学员**优化的版本:
- 数据源用 **akshare**(国内可访问、零 VPN、免注册),取代了视频里的 yfinance
- 标的尽量保持原意:美股 ETF→A 股 ETF / 国际公司→A 股龙头
- 所讲的**概念和方法 100% 一致**,但**具体数字可能与视频里略有差异**(因为是不同时间窗 / 不同标的)
- 一般情况国内 `pip install akshare` 即可,无需 token / VPN

**课件生成日期:** 2026-05-29  ·  **建议学习时长:** 22 分钟

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有必需的 Python 包(含 `akshare`),缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续


In [4]:
# === 环境自检 + 自动安装(运行此单元格即可)===
import importlib, subprocess, sys, os

REQUIRED = ["akshare", "baostock", "matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels"]
PIP_NAME = {"sklearn":"scikit-learn","cv2":"opencv-python","PIL":"Pillow","bs4":"beautifulsoup4","yaml":"PyYAML"}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))
if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置 ===
import matplotlib, matplotlib.pyplot as plt, matplotlib.font_manager as fm
CJK = ["/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
       "C:/Windows/Fonts/msyh.ttc","C:/Windows/Fonts/simhei.ttf",
       "/System/Library/Fonts/PingFang.ttc","/System/Library/Fonts/STHeiti Medium.ttc"]
for p in CJK:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP","Microsoft YaHei","PingFang SC","SimHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪")


✓ 所有 9 个必需库已就绪
✓ 中文字体已加载:NotoSansCJK-Regular.ttc
✓ 环境就绪


## 🔌 第二步:加载国内数据助手

下面这一格是**工具函数**(可以折叠,不需要修改)。它把 `yfinance` 风格的 ticker(如 `600519.SS`)自动路由到对应的 akshare 接口,提供 `get_close(ticker)` 和 `get_close_multi(tickers_dict)` 两个函数。

In [5]:
# === 国内数据源助手(akshare 后端,不需要 VPN)===
# 这一格是工具函数,可以折叠,不需要修改。
# 它把 yfinance 风格的 ticker(如 "600519.SS" / "0700.HK" / "AAPL" / "BTC-USD")
# 自动路由到对应的 akshare 接口,统一返回 yfinance 风格的 Close DataFrame。

import re
from datetime import datetime, timedelta
import pandas as pd
import akshare as ak

_TICKER_MAP = {
    "^GSPC": ("us_index_sina", ".INX"),
    "^DJI":  ("us_index_sina", ".DJI"),
    "^IXIC": ("us_index_sina", ".IXIC"),
    "GC=F":  ("foreign_futures", "GC"),
    "SI=F":  ("foreign_futures", "SI"),
    "CL=F":  ("foreign_futures", "CL"),
    "BTC-USD": ("crypto", "BTC"),
    "ETH-USD": ("crypto", "ETH"),
}

def _retry(fn, *args, _retries=4, _wait=1.5, **kwargs):
    """akshare 上游(东方财富/新浪/Binance)偶有 RemoteDisconnected / Timeout,自动重试 4 次。
    2026-05-11 加:用户跑 cn notebook 拉 002594.SZ 时上游断连 → 整节卡死。
    每次重试间隔 _wait 秒(指数退避 1x → 1.5x → 2.25x)。
    """
    import time as _t
    last_err = None
    wait = _wait
    for i in range(_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            name = type(e).__name__
            if i == _retries - 1:
                print(f"  ✗ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次仍失败({name}),放弃")
                raise
            print(f"  ⚠ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次失败({name}),{wait:.1f}s 后重试")
            _t.sleep(wait)
            wait *= 1.5

def _parse_period(period):
    end = datetime.today()
    m = re.match(r"^(\d+)\s*(y|mo|d|w)$", period.lower().strip())
    days = 365 * 3 if not m else int(m.group(1)) * {"y":365,"mo":30,"w":7,"d":1}[m.group(2)]
    return (end - timedelta(days=days+30)).strftime("%Y%m%d"), end.strftime("%Y%m%d")

def _classify(ticker):
    t = ticker.strip()
    if t in _TICKER_MAP: return _TICKER_MAP[t]
    if t.endswith((".SS",".SH",".SZ")):
        code = t.split(".")[0]
        if code.startswith(("51","159","58")) or code in ("510300","510500","510050","511010","513100"):
            return ("a_etf", code)
        if code in ("000300","000016","000905","000852","000001"):
            return ("a_index", code)
        return ("a_stock", code)
    if t.endswith(".HK"):
        return ("hk", t.split(".")[0].zfill(5))
    return ("us", t)

def _norm(df, dc, cc):
    out = df[[dc, cc]].copy()
    out[dc] = pd.to_datetime(out[dc])
    return out.set_index(dc).sort_index()[cc].astype(float).rename("Close")

def get_close(ticker, period="3y"):
    """返回某标的 Close 价格 series。后端 akshare,中国可访问。
    所有 ak.* 调用都过 _retry(4 次,指数退避)— 防东方财富/新浪上游瞬时断连。
    """
    start, end = _parse_period(period)
    kind, sym = _classify(ticker)
    if kind == "a_stock":
        return _norm(_retry(ak.stock_zh_a_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_etf":
        return _norm(_retry(ak.fund_etf_hist_em, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_index":
        idx_map = {"000300":"sh000300","000016":"sh000016","000905":"sh000905","000852":"sh000852","000001":"sh000001"}
        s = _norm(_retry(ak.stock_zh_index_daily_em, symbol=idx_map.get(sym, f"sh{sym}")), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "hk":
        return _norm(_retry(ak.stock_hk_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "us":
        s = _norm(_retry(ak.stock_us_daily, symbol=sym, adjust="qfq"), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "us_index_sina":
        s = _norm(_retry(ak.index_us_stock_sina, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "foreign_futures":
        s = _norm(_retry(ak.futures_foreign_hist, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "crypto":
        import requests as _rq
        def _binance():
            r = _rq.get("https://api.binance.com/api/v3/klines",
                        params={"symbol": f"{sym}USDT", "interval": "1d", "limit": 1000}, timeout=15)
            r.raise_for_status()
            return r.json()
        klines = _retry(_binance)
        df = pd.DataFrame(klines, columns=["open_time","open","high","low","close","volume",
                                            "close_time","qav","trades","tbb","tbq","ignore"])
        df["date"] = pd.to_datetime(df["open_time"], unit="ms")
        df["close"] = df["close"].astype(float)
        s = df.set_index("date").sort_index()["close"].rename("Close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    raise ValueError(f"unsupported ticker: {ticker}")

def get_close_multi(tickers, period="3y"):
    """批量取 Close,返回 DataFrame,列名是 tickers dict 的 key(中文名),按交集日期对齐。"""
    series = {name: get_close(t, period=period) for name, t in tickers.items()}
    return pd.concat(series, axis=1).sort_index()

print("✓ cn_data 助手已加载 — 用 get_close(ticker) / get_close_multi(tickers_dict) 拉数据")


✓ cn_data 助手已加载 — 用 get_close(ticker) / get_close_multi(tickers_dict) 拉数据


## 学习目标

- 掌握 DatetimeIndex 时间索引 — 它支持「部分字符串索引」,写一句日期或月份就能筛出对应数据,还能直接从索引取出小时、分钟做日内分析
- 学会 pd.to_datetime — 把各种格式的日期字符串统一解析成标准时间戳,理解格式参数的作用和月日顺序这类常见解析坑
- 吃透 resample 重采样 — 它是「时间版的 groupby」,能把分钟聚合成日线、日线聚合成周线月线,是量化处理行情最高频的操作
- 记牢 OHLCV 聚合规则 — 开盘取第一个、最高取最大、最低取最小、收盘取最后一个、成交量要求和,绝不能所有列一刀切取平均,这是新手最大的坑
- 理解时区处理 — tz_localize 是给时间贴时区标签、tz_convert 是换算到另一个时区看同一时刻,做 A 股加美股的跨市场对齐前必须先统一时区

## 历史背景:小陈把成交量也取了平均,量价信号全错还浑然不觉

讲一个量化新手小陈的真实翻车。小陈想做一个「放量突破」策略 — 当某天成交量明显放大、同时价格突破近期高点时买入。他手里有茅台的5 分钟数据,需要先把它聚合成日线,才能算「每天的成交量」和「每天的收盘价」。

他知道 Pandas 有个 resample 函数能做聚合,于是图省事,写了一句 df.resample('D').mean()—— 把所有列都取了平均。代码不报错,跑得很顺,他就以为大功告成了。

问题大了。「取平均」对成交量是致命错误。一天有几十根5 分钟 K 线,每根都有自己的成交量,一天真实的总成交量应该是把这几十根的量全部加起来(求和),而他取了平均,等于把总量除以了几十,数字一下子小了几十倍。更糟的是,收盘价取平均也错了 —— 一天的收盘价应该是最后一根 K 线的价格,而不是全天价格的平均值。结果他算出来的「每天成交量」全是缩水几十倍的假数据,「每天收盘价」也是糊成一团的平均值。

基于这堆错数据,他的「放量突破」策略自然永远触发不了正确信号,回测结果一塌糊涂。他查了3 天代码,逻辑部分一个 bug 都没有,百思不得其解,最后才发现根子在最开始那一句 resample 用错了聚合方式。

这个故事的教训极其重要:做时间重采样时,不同的列必须用不同的聚合方式。这套规则在金融里有个标准名字,叫 OHLCV 聚合 —— 开盘价 Open 取这段时间的第一个、最高价 High 取最大、最低价 Low 取最小、收盘价 Close 取最后一个、成交量 Volume 要求和。价格类绝不能取平均,成交量绝不能取最后一个。一刀切是新手最常见、也最隐蔽的错误,因为代码不报错,你不盯着结果根本发现不了。

今天我们就用真实的茅台5 分钟数据,把正确的 OHLCV 聚合亲手跑一遍,再故意演示一遍「全取平均」的错误后果,让你对这个坑刻骨铭心。学完这一节,你就掌握了把任意高频数据转成任意周期 K 线的核心能力,这是写任何策略的第一步。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. DatetimeIndex — 给时间装上超能力的专用索引

昨天我们的 DataFrame 行标签虽然是日期,但只是把它当普通文字标签用。今天要把日期升级成 DatetimeIndex —— Pandas 专门为时间打造的索引类型,一旦升级,会解锁一堆超能力。

第一个超能力叫「部分字符串索引」。普通索引你要取某天数据,得精确报出完整标签。而 DatetimeIndex 聪明得多:你写 df.loc['2024-12'],它自动把整个 12 月的所有行都筛出来;你写 df.loc['2024-12-02'],它把那一天的所有分钟 K 线都筛出来。给个年份选一整年、给个月份选一整月、给一天选一整天,完全符合人的直觉,一句话搞定,不用写复杂的筛选条件。

第二个超能力是「时间属性直取」。DatetimeIndex 的每个时间点都知道自己是几年几月几号、星期几、几点几分。你可以直接从索引里取出小时(index.hour)、星期几(index.dayofweek),用来做日内分析或者剔除周末。比如想看「每天开盘第一个小时的成交量」,直接按小时分组就行。

第三个超能力是「时间感知的运算」,也就是后面要讲的 resample 和时区转换,全都建立在 DatetimeIndex 之上。普通文字索引做不了这些。

怎么得到 DatetimeIndex?把你的日期/时间列用 pd.to_datetime 解析一下,再 set_index 设成索引就行。这是处理任何行情数据的标准第一步。记住:行情数据进门,先把时间列转成 DatetimeIndex,后面一切才方便。


### 2. pd.to_datetime — 把五花八门的时间字符串理顺

真实世界的时间数据格式乱得很。有的写成 2024-12-02,有的写成 2024/12/02,有的写成 12/02/2024,还有的像 baostock 这样写成一长串数字 20241202093500000(年月日时分秒毫秒糊在一起)。pd.to_datetime 就是把这些五花八门的字符串,统一解析成 Pandas 能理解的标准时间戳。

大多数常见格式,pd.to_datetime 能自动识别,直接喂给它就行。但碰到不标准的格式(比如那串糊在一起的数字),就需要用 format 参数明确告诉它怎么拆 —— 哪几位是年、哪几位是月、哪几位是时分秒。指定了 format 不仅解析更准,速度也快很多,处理大数据时差别明显。

这里有个著名的坑要提醒:月和日的顺序。同样是 03/04 这个写法,美式习惯是「3 月 4 号」(月在前),而很多地方是「4 月 3 号」(日在前)。如果不指定格式让 Pandas 自己猜,它可能猜错,把月当成日、日当成月,而且不报错,数据就这么悄悄错了。所以碰到有歧义的格式,一定要用 format 参数或者 dayfirst 参数明确指定,别赌它能猜对。

一句话:时间字符串进门先 to_datetime 解析成时间戳;格式怪或有歧义的,显式给 format,既准又快。这是所有时间处理的起点。


### 3. resample 重采样 — 时间版的 groupby

resample 是今天的绝对主角,也是量化处理行情数据用得最多的操作之一。一句话理解它:resample 就是「时间版的 groupby 分组」。

普通的 groupby 是按某个类别分组(比如按行业把股票分成几堆)。而 resample 是按时间段分组 —— 把数据按你指定的周期(每天、每周、每月)切成一段一段,然后对每一段做聚合。最典型的用途就是「降频」:把5 分钟的高频数据,聚合成日线;把日线聚合成周线、月线。这正是开头小陈要做的事。

用法很直观:df.resample('D') 表示「按天分组」,'W' 是按周,'ME' 是按月末。分组之后,跟 groupby 一样要告诉它每组怎么聚合 —— 求和用 sum、取最后一个用 last、取最大用 max。比如把分钟收盘价聚合成日收盘价,就是 df['close'].resample('D').last(),意思是「每天取最后一根 K 线的收盘价」。

几个实用细节。第一,resample 按自然周期切,周末、节假日这些没数据的日子会产生空行(NaN),记得用 dropna 清掉。第二,周期标签有讲究,'ME' 是月末对齐、'MS' 是月初对齐,用错了日期会差半个月,要按需求选。第三,resample 还能「升频」(把日线插值成分钟),但量化里很少用,主要还是降频。

掌握了 resample,你就能把任意高频数据转成任意周期的 K 线,这是做任何周期策略的基本功。但用它有个大前提 —— 不同的列要用不同的聚合方式,这就是下一个概念。


### 4. OHLCV 聚合规则 — 不同列必须用不同的聚合方式

这是今天最重要、也最容易踩的实战要点,开头小陈翻车的根源就在这里。当你把高频数据 resample 成低频时,行情数据的五个标准字段 —— 开盘价 Open、最高价 High、最低价 Low、收盘价 Close、成交量 Volume(合称 OHLCV)—— 每一个都有它专属的、不能搞错的聚合方式。

逐个来说。开盘价 Open 取这段时间的「第一个」值 —— 一天的开盘价就是当天第一根 K 线的开盘价,用 first。最高价 High 取「最大」值 —— 一天的最高价是当天所有 K 线里最高的那个,用 max。最低价 Low 取「最小」值,用 min。收盘价 Close 取「最后一个」值 —— 一天的收盘价是当天最后一根 K 线的收盘价,用 last。成交量 Volume 取「求和」—— 一天的总成交量是当天所有 K 线成交量的总和,用 sum。

为什么不能一刀切全取平均?因为含义完全不对。价格取平均,你得到的是个全天均价,既不是开盘也不是收盘,任何 K 线图都画不出来,所有基于开收盘的信号全废。成交量取平均更离谱,真实的日成交量应该是几十根 K 线的总和,取了平均等于凭空缩水几十倍,放量缩量的信号彻底反过来。这正是小陈的策略永远不触发的原因。

实战里怎么做?用 resample 配合 agg,把每一列对应的聚合方式写成一个字典:开盘 first、最高 max、最低 min、收盘 last、成交量 sum。一次性正确聚合所有列。记住这套 OHLCV 规则,它是处理行情数据的铁律,刻进肌肉记忆 —— 价格分四种各管各的,成交量永远求和,绝不一刀切取平均。


### 5. 时区与工作日偏移 — 跨市场对齐的隐形地基

最后讲时区。平时只做 A 股可能感觉不到时区的存在,但一旦你要做跨市场分析 —— 比如研究 A 股和美股的联动 —— 时区就成了一个绕不开的隐形地基,处理不好会得出完全错误的结论。

核心是两个容易混淆的操作。第一个是 tz_localize,「贴时区标签」。刚解析出来的时间是「裸」的(naive),不带任何时区信息,只是一串钟点数字。tz_localize 的作用是告诉 Pandas「这串钟点是哪个时区的」,比如 A 股数据全是北京时间,就 tz_localize('Asia/Shanghai')。注意它只是贴个标签,钟点数字本身不变。第二个是 tz_convert,「换算时区」。它把一个已经带时区的时间,换算成另一个时区下「同一时刻」的钟点。比如北京时间上午9:30,换算成纽约时间就成了前一天的晚上8:30(钟点变了,但指的是宇宙中同一个瞬间)。

这俩为什么重要?做 A 股加美股的跨市场对齐时,如果不统一时区,你会把北京时间的9:30和纽约时间的9:30当成同一时刻去对齐,实际上它们差了十几个小时,数据全错位 —— 这跟之前讲过的跨市场对齐陷阱一脉相承。正确做法是:两边数据各自 localize 到自己的时区,再统一 convert 到同一个时区(比如都转成 UTC),然后才能放心对齐。

顺带讲一个实用的小工具:工作日偏移 BusinessDay。金融计算经常要「往后推一个交易日」,但不能简单加一天(可能撞上周末)。pd.offsets.BDay 能自动跳过周末,帮你正确地在工作日之间移动。这在算「T 加一交割」「信号次日执行」这类逻辑时特别有用。一句话:localize 贴标签、convert 换时区,跨市场先统一时区;工作日移动用 BDay 自动跳周末。


## 实操:时间索引与重采样五件套 — to_datetime/DatetimeIndex/resample/OHLCV/时区(中国版 — baostock,跟原版相同)

下面这段代码用 akshare 抓数据,国内零 VPN 跑通。**直接 Run All** 看结果。

**依赖:** `pip install pandas numpy matplotlib akshare statsmodels scipy`

In [6]:
# day_035_datetime_resample.py — 时间索引与重采样:DatetimeIndex/to_datetime/resample/OHLCV 聚合/时区(中国版 — baostock,跟原版相同)
# 用真实 A 股 5 分钟线,把分钟数据 resample 成日线 / 周线,讲透时间是金融数据的灵魂
# 数据:baostock 5 分钟线(免费、稳定、国内零翻墙;A 股原生北京时间)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import baostock as bs

pd.set_option('display.width', 140)
CODE = 'sh.600519'                  # 茅台,用一只票把分钟线讲透
NAME = '茅台'
START = '2024-11-01'
END = '2024-12-31'

# ============ 0. 数据拉取:5 分钟线 OHLCV ============
print('==== 0. 数据拉取(baostock 5 分钟线)====')
lg = bs.login()
if lg.error_code != '0':
    raise RuntimeError(f'baostock 登录失败:{lg.error_msg}')
rs = bs.query_history_k_data_plus(
    CODE, 'date,time,open,high,low,close,volume',
    start_date=START, end_date=END, frequency='5', adjustflag='2')
rows = []
while (rs.error_code == '0') and rs.next():
    rows.append(rs.get_row_data())
bs.logout()
raw = pd.DataFrame(rows, columns=['date', 'time', 'open', 'high', 'low', 'close', 'volume'])
print(f'{NAME} 5 分钟原始数据 {len(raw)} 条')

# ============ 1. pd.to_datetime:把字符串解析成时间戳 ============
print('\n==== 1. pd.to_datetime 解析时间 ====')
# baostock 的 time 字段形如 '20241101093500000'(年月日时分秒毫秒)
print('原始 time 字段样例:', raw['time'].iloc[0])
ts = pd.to_datetime(raw['time'], format='%Y%m%d%H%M%S%f')
df = raw.copy()
for c in ['open', 'high', 'low', 'close', 'volume']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = df.set_index(ts).drop(columns=['date', 'time']).sort_index()
df.index.name = '时间'
print(f'解析后索引类型 = {type(df.index).__name__}')
print(f'时间范围 {df.index.min()} → {df.index.max()}')

# ============ 2. DatetimeIndex:部分字符串索引 + .时间属性 ============
print('\n==== 2. DatetimeIndex 的超能力 ====')
# 部分字符串索引:给一个日期就选出那天所有分钟
one_day = df.loc['2024-12-02']
print(f'df.loc["2024-12-02"] 选出一整天 {len(one_day)} 根 5 分钟 K 线(部分字符串索引)')
# 给一个月份选出整月
one_month = df.loc['2024-12']
print(f'df.loc["2024-12"] 选出整个 12 月 {len(one_month)} 根 K 线')
# 直接从索引拿小时/分钟,做日内分析
print('索引自带时间属性 index.hour 样例:', sorted(set(df.index.hour)))

# ============ 3. resample:时间版的 groupby(分钟 → 日线)============
print('\n==== 3. resample 分钟 → 日线 ====')
# 核心:不同列聚合方式不同!开=first 高=max 低=min 收=last 量=sum
daily = df.resample('D').agg({
    'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last', 'volume': 'sum',
}).dropna(how='any')
print(f'{len(df)} 根 5 分钟线  →  {len(daily)} 根日线')
print('日线前 3 行:')
print(daily.head(3).round(2))
# 校验:某天日线的 high 必须等于那天所有分钟 high 的最大值
chk_day = '2024-12-02'
minute_high = df.loc[chk_day, 'high'].max()
daily_high = daily.loc[chk_day, 'high']
print(f'校验 {chk_day}:分钟最高 {minute_high:.2f} == 日线 high {daily_high:.2f} ?', np.isclose(minute_high, daily_high))

# ============ 4. OHLCV 聚合规则:价格不能取平均、量不能取 last ============
print('\n==== 4. 聚合规则对错对比 ====')
# 错误示范:对所有列一律取平均
wrong = df.resample('D').mean().dropna(how='any')
# 对比某天:正确收盘(last) vs 错误收盘(mean)
right_close = daily.loc[chk_day, 'close']
wrong_close = wrong.loc[chk_day, 'close']
print(f'{chk_day} 收盘价 — 正确(取最后一根)= {right_close:.2f}')
print(f'{chk_day} 收盘价 — 错误(取全天平均)= {wrong_close:.2f}  ← 差了 {abs(right_close-wrong_close):.2f} 元,信号全错')
# 成交量:必须 sum 不能 mean
vol_sum = df.loc[chk_day, 'volume'].sum()
vol_mean = df.loc[chk_day, 'volume'].mean()
print(f'{chk_day} 成交量 — 正确(求和)= {vol_sum:,.0f}  错误(平均)= {vol_mean:,.0f}(差 {vol_sum/vol_mean:.0f} 倍)')

# resample 到周线
weekly = df['close'].resample('W').last().dropna()
print(f'再 resample 到周线:{len(weekly)} 根')

# ============ 5. 时区:tz_localize 贴标签 vs tz_convert 换时区 ============
print('\n==== 5. 时区处理 ====')
naive = df.index[:3]
print('原始时间不带时区(naive):', list(naive.astype(str)))
# A 股是北京时间,先「贴上」时区标签(localize,不改钟点只贴标签)
aware = df.index.tz_localize('Asia/Shanghai')
print('贴上北京时区后:', str(aware[0]))
# 换算成美东时间「同一时刻」(convert,钟点会变)
us = aware.tz_convert('America/New_York')
print('换算成纽约时间(同一时刻):', str(us[0]))
print('教训:localize 只贴标签不改钟点;convert 才真正换算钟点。跨市场对齐前必须统一时区')
# BusinessDay:跳过周末的工作日偏移
last_day = daily.index[-1]
next_bday = last_day + pd.offsets.BDay(1)
print(f'最后交易日 {last_day.date()} 后推 1 个工作日 = {next_bday.date()}(自动跳过周末)')

# ============ 6. 出图 ============
# chart_01:某一天的分钟级日内走势
intraday = df.loc[chk_day, 'close']
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(intraday.index, intraday.values, color='#1f77b4', lw=1.3)
ax.set_title(f'{NAME} {chk_day} 日内 5 分钟收盘价走势')
ax.set_ylabel('价格(元)'); ax.grid(alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout(); fig.savefig('chart_intraday.png', dpi=130); plt.close(fig)

# chart_02:分钟 close 全貌 + resample 后的日线 close 叠加(点)
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(df.index, df['close'], color='#aaaaaa', lw=0.6, label='5 分钟 close')
ax.plot(daily.index, daily['close'], 'o-', color='#d62728', ms=4, lw=1.2, label='日线 close(resample)')
ax.set_title(f'{NAME}:分钟数据 resample 成日线(2024-11 ~ 12)')
ax.set_ylabel('价格(元)'); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig('chart_resample.png', dpi=130); plt.close(fig)

# chart_03:日内成交量分布(按时段聚合,体现 volume=sum 的意义 + 日内 U 型)
by_time = df.groupby(df.index.strftime('%H:%M'))['volume'].mean()
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(by_time)), by_time.values, color='#2ca02c')
step = max(1, len(by_time)//12)
ax.set_xticks(range(0, len(by_time), step))
ax.set_xticklabels(by_time.index[::step], rotation=45, ha='right')
ax.set_title(f'{NAME} 日内各时段平均成交量(经典 U 型:开盘收盘量大)')
ax.set_ylabel('平均成交量'); ax.grid(alpha=0.3, axis='y')
fig.tight_layout(); fig.savefig('chart_intraday_volume.png', dpi=130); plt.close(fig)

# chart_04:日线 vs 周线收盘价对比
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(daily.index, daily['close'], '-', color='#1f77b4', lw=1, label='日线')
ax.plot(weekly.index, weekly.values, 's-', color='#ff7f0e', ms=6, lw=1.5, label='周线')
ax.set_title(f'{NAME}:同一段行情 日线 vs 周线(resample 到不同频率)')
ax.set_ylabel('收盘价(元)'); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig('chart_daily_weekly.png', dpi=130); plt.close(fig)

print('\n[done] 4 张图已保存,output.txt 见上方全部打印')

==== 0. 数据拉取(baostock 5 分钟线)====
login success!
logout success!
茅台 5 分钟原始数据 2064 条

==== 1. pd.to_datetime 解析时间 ====
原始 time 字段样例: 20241101093500000
解析后索引类型 = DatetimeIndex
时间范围 2024-11-01 09:35:00 → 2024-12-31 15:00:00

==== 2. DatetimeIndex 的超能力 ====
df.loc["2024-12-02"] 选出一整天 48 根 5 分钟 K 线(部分字符串索引)
df.loc["2024-12"] 选出整个 12 月 1056 根 K 线
索引自带时间属性 index.hour 样例: [9, 10, 11, 13, 14, 15]

==== 3. resample 分钟 → 日线 ====
2064 根 5 分钟线  →  43 根日线
日线前 3 行:
               open     high      low    close   volume
时间                                                     
2024-11-01  1444.20  1467.00  1444.20  1456.31  3255500
2024-11-04  1466.92  1471.72  1450.94  1470.03  2927200
2024-11-05  1462.24  1497.65  1458.46  1497.37  4590100
校验 2024-12-02:分钟最高 1452.54 == 日线 high 1452.54 ? True

==== 4. 聚合规则对错对比 ====
2024-12-02 收盘价 — 正确(取最后一根)= 1448.00
2024-12-02 收盘价 — 错误(取全天平均)= 1447.01  ← 差了 0.99 元,信号全错
2024-12-02 成交量 — 正确(求和)= 2,684,000  错误(平均)= 55,917(差 48 倍)
再 resample 到周线:10 根

==== 5. 时区处理 ====
原始

## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| A 股散户 | 茅台 等任意个股的分钟数据 | 散户做日内看的是分钟图,但要做策略回测、算「每天涨跌」「每天成交量」,必须先把分钟数据 resample 成日线。这一步聚合方式一旦搞错(比如成交量取了平均),后面所有量价信号全废,而且代码不报错,极其隐蔽。这正是这一节故事的原型,也是每个量化新手都要趟过的坑。 |
| 量化研究 / 数据工程 | 全市场 tick / 分钟行情库 | 量化机构的数据团队拿到的是最原始的逐笔(tick)或分钟数据,要按需聚合成 1 分钟、5 分钟、15 分钟、日、周等各种周期 K 线供策略调用。resample 加 OHLCV 聚合就是这套数据流水线的核心环节,聚合规则写错一次,下游几十个策略全部中招。 |
| 跨市场 / 全球宏观 | A 股 + 美股 + 港股联动 | 研究 A 股跟美股隔夜的联动关系时,A 股是北京时间、美股是纽约时间,差十几个小时。不先用 tz_localize 和 tz_convert 把两边时区统一,就会把不同时刻的数据错误对齐,算出来的相关性、领先滞后关系全是假的。这是跨市场分析最隐蔽的地基性错误。 |
| 量价分析 / VWAP | 任意活跃股的日内数据 | 成交量在一天内呈现经典的「U 型」—— 开盘和收盘前量大、午盘量小。算成交量加权均价 VWAP、判断放量缩量,都依赖对成交量的正确聚合(求和)。按日内时段对成交量分组求平均,就能看出这个 U 型,这是理解市场微观结构的入门一课。 |
| 事件 / 财报统计 | 指数或个股的长周期数据 | 研究「每月最后一个交易日效应」「季度财报季的波动」等日历效应时,要把日线数据按月('ME')或按季 resample,统计每个周期的收益、波动。这时月末对齐('ME')还是月初对齐('MS')必须选对,否则把财报日归错月份,统计结论就偏了。 |


## 常见坑

### ⚠ 01. resample 所有列一刀切取平均 — OHLCV 全错且不报错

对行情数据 resample 时图省事写 .mean() 把所有列都取平均,是最隐蔽的致命错。价格取平均得到的是无意义的全天均价,成交量取平均凭空缩水几十倍,而代码完全不报错。**正确做法**:用 .agg() 配字典,开盘 first、最高 max、最低 min、收盘 last、成交量 sum,每列各管各的,这套 OHLCV 规则要刻进肌肉记忆。

### ⚠ 02. 忘了 dropna — resample 出一堆周末空行

resample 按自然周期切分,周末、节假日没有交易数据,会产生一堆全是 NaN 的空行。不清理的话,后续画图断断续续、算指标被 NaN 传染。**正确做法**:resample 聚合后顺手 .dropna(how='any') 或 how='all',把没有真实交易的空周期清掉,只留有数据的交易日。

### ⚠ 03. tz_localize 和 tz_convert 混用 — 时间错十几个小时

把贴标签的 tz_localize 和换算的 tz_convert 搞混:本该 localize(只贴标签不改钟点)却用了 convert(改了钟点),时间一下错十几个小时。还有 naive(不带时区)和 aware(带时区)的时间直接相减会报错。**正确做法**:裸时间先 localize 贴上本时区,要跨时区比较再 convert;同一组运算里所有时间的时区状态要一致。

### ⚠ 04. to_datetime 月日顺序猜错 — 数据悄悄串位

碰到 03/04 这种有歧义的格式,不指定格式让 Pandas 自己猜,它可能把 3 月 4 号猜成 4 月 3 号,而且不报错。一旦猜错,整个时间轴串位,后续全错。**正确做法**:格式有歧义时,显式传 format 参数(如 '%m/%d/%Y')或 dayfirst 参数明确告诉它月日顺序,绝不靠猜。

### ⚠ 05. resample 周期标签用错 — 月末月初差半个月

按月 resample 时,'ME' 是对齐到月末最后一天、'MS' 是对齐到月初第一天,'W' 默认对齐到周日。标签选错,聚合出来的日期会整体偏移半个月或几天,把数据归错周期。**正确做法**:想清楚要月末还是月初对齐,查准频率别名;聚合后打印几行索引,确认日期落在预期位置。

## 实战 SOP · 时间索引与重采样实战 7 条 SOP

1. 行情数据进门第一步:时间列 pd.to_datetime 解析 + set_index 成 DatetimeIndex — 一切时间操作的前提
2. 格式怪或有歧义的时间显式给 format 参数 — 又准又快,杜绝月日顺序猜错
3. resample 降频是量化最高频操作 — 'D' 按天、'W' 按周、'ME' 按月末,想清楚对齐到哪
4. OHLCV 聚合用 .agg 配字典 — 开 first / 高 max / 低 min / 收 last / 量 sum,绝不一刀切 mean
5. resample 后顺手 dropna — 清掉周末节假日产生的空周期
6. 跨市场对齐前先统一时区 — 各自 tz_localize 到本时区,再 tz_convert 到同一时区
7. 往后推交易日用 pd.offsets.BDay 不用加一天 — 自动跳过周末,T+1 交割/次日执行都靠它

> 把这段打印贴在你电脑边。

## 总结 · 你应该带走的

2. DatetimeIndex 是时间专用索引 — 支持部分字符串索引(给月份选整月、给一天选一天),还能直取小时/星期做日内分析
3. pd.to_datetime 把各种时间字符串解析成标准时间戳 — 格式怪或有歧义显式给 format,杜绝月日顺序猜错
4. resample 是时间版 groupby — 按周期把数据切段聚合,最常用于把分钟降频成日线/周线/月线
5. OHLCV 聚合规则是铁律 — 开 first / 高 max / 低 min / 收 last / 量 sum,价格不能取平均、成交量不能取最后一个
6. 一刀切取平均是最隐蔽的坑 — 代码不报错但量价信号全废,小陈的策略永远不触发就栽在这
7. 时区:tz_localize 贴标签不改钟点、tz_convert 换算钟点看同一时刻 — 跨市场对齐前必须统一时区
8. 工作日偏移用 pd.offsets.BDay — 往后推交易日自动跳过周末,T+1 交割、信号次日执行靠它

## 自测题

**Q1.** DatetimeIndex 比普通文字索引多了哪些超能力?写 df.loc['2024-12'] 会选出什么?(提示:部分字符串索引、时间属性直取、支持 resample/时区;选出整个 12 月的所有行)

**Q2.** resample 是什么?它跟 groupby 有什么类比关系?把分钟收盘价聚合成日收盘价该怎么写?(提示:时间版 groupby,按时间段分组聚合;df['close'].resample('D').last())

**Q3.** OHLCV 五个字段各自该用什么聚合方式?为什么成交量不能取平均?(提示:开 first/高 max/低 min/收 last/量 sum;日成交量是当天所有 K 线之和,取平均会缩水几十倍)

**Q4.** tz_localize 和 tz_convert 有什么区别?做 A 股加美股跨市场对齐时为什么必须先处理时区?(提示:localize 贴标签不改钟点、convert 换算钟点;不统一时区会把差十几小时的数据错误对齐)

**Q5.** 为什么 resample 之后通常要 dropna?往后推一个交易日为什么不能简单加一天?(提示:周末节假日产生空周期;加一天可能撞周末,要用 BDay 自动跳过)

把答案写下来,3 天后再回看。

## 下一节预告

**Day 036 · 滚动窗口 rolling** (Rolling Windows)

第 36 课 滚动窗口 rolling。今天学会了把数据聚合成不同周期,明天讲怎么在时间序列上「开一个移动的窗口」做计算 —— 这就是 rolling。你天天听到的 20 日均线、布林带,本质都是 rolling 滚动窗口算出来的。我们会讲 rolling 滚动、expanding 累计、ewm 指数加权三兄弟,亲手实现均线和布林带,还有怎么用 min_periods 处理开头的 NaN。这是所有技术指标的共同地基,学完你就能自己造任何指标。

## 推荐阅读

- Wes McKinney《Python for Data Analysis》(2022 第 3 版,O'Reilly)— 第 11 章专讲时间序列、DatetimeIndex、resample 与时区,Pandas 作者亲笔,本节最对口的权威章节
- Pandas 官方文档《Time series / date functionality》(免费在线,pandas.pydata.org)— resample 频率别名、offset、时区处理的权威参考,频率字符串('D'/'W'/'ME')查这里最准
- Jake VanderPlas《Python Data Science Handbook》(2016,jakevdp.github.io 免费在线)— 第 3 章 Working with Time Series 一节,用真实数据讲 resample,通俗易懂
- Marcos López de Prado《Advances in Financial Machine Learning》(2018,Wiley)— 第 2 章讲金融数据的「信息驱动 Bar」(成交量 Bar、美元 Bar),是 resample 思想在量化里的高阶延伸
- Python 工具栈 — Pandas(resample 核心)、pytz / zoneinfo(时区数据库)、pandas-market-calendars(各国交易所交易日历,处理节假日)、mplfinance(画 K 线图),这四个是时间序列处理的进阶生态